<a href="https://colab.research.google.com/github/ShivaniMareddy/GPT/blob/main/GPT_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 1: Data Preparation

*Objective*

Prepare text data for GPT training.

Load dataset
Convert to lowercase
Remove unwanted symbols
Tokenize text
Build vocabulary
Create input-output sequences

Example:
Input: 'the cat chased'
Target: 'the'

In [ ]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer

# Load dataset
texts = [
    "The cat chased the mouse.",
    "The dog barked loudly.",
    "Machine learning is powerful.",
    "Deep learning uses neural networks."
]

# Convert to lowercase and remove punctuation
texts = [re.sub(r'[^a-zA-Z\s]', '', t.lower()) for t in texts]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

sequences = []

# Create input-output sequences
for text in texts:
    token_list = tokenizer.texts_to_sequences([text])[0]

    for i in range(1, len(token_list)):
        input_seq = token_list[:i]
        target = token_list[i]

        sequences.append((input_seq, target))

print("Vocabulary Size:", vocab_size)

print("\nFirst 5 Sequences:")
for seq in sequences[:5]:
    print(seq)

Vocabulary Size: 16

First 5 Sequences:
([1], 3)
([1, 3], 4)
([1, 3, 4], 1)
([1, 3, 4, 1], 5)
([1], 6)


#Task 2: Token Embeddings

*Objective*

Convert token IDs into dense vectors.

Embeddings capture semantic meaning unlike one-hot vectors.

In [ ]:
import tensorflow as tf
embedding_dim=32
embedding_layer=tf.keras.layers.Embedding(vocab_size,embedding_dim)

sample_input=tf.constant([1,2,3])
embedded=embedding_layer(sample_input)

print("Embeddings",embedded)
print("Embedding Shape",embedded.shape)


Embeddings tf.Tensor(
[[ 1.39959492e-02  1.03170760e-02  2.27571838e-02 -1.67683251e-02
  -4.02970202e-02 -2.92491913e-02 -1.49451867e-02  2.32207291e-02
   1.80651434e-02 -3.66971269e-02 -4.14766371e-04 -1.44927613e-02
  -2.07016002e-02 -8.26289505e-03 -7.29737431e-03 -2.26636175e-02
   4.85952832e-02 -3.71782780e-02  2.07995065e-02  2.61528157e-02
   4.32617962e-05 -1.47229433e-02 -4.21325341e-02  2.81606428e-02
   2.14073099e-02 -3.64107266e-02 -9.49639082e-03 -1.74022838e-03
  -2.02841517e-02  1.74410604e-02  4.07900661e-03 -4.11233306e-02]
 [-1.34198666e-02 -2.63695121e-02 -4.75436933e-02 -2.38975044e-02
  -3.58203873e-02  3.93116735e-02 -2.78053768e-02 -4.06165048e-03
  -3.49122770e-02  1.46024935e-02  2.14193016e-03 -4.63465452e-02
   4.30654623e-02 -3.45284343e-02 -3.30945253e-02  3.74338739e-02
  -3.51992361e-02  1.14493258e-02  3.29689272e-02 -1.31736994e-02
   1.95670873e-04  3.91957872e-02  2.99181789e-03 -2.78035998e-02
   4.66514118e-02 -8.80751759e-03 -1.88357960e-02 -9.

#Positional Encoding

In [ ]:
import numpy as np

def positional_encoding(max_len, d_model):
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angles = pos * angle_rates

    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])

    return pe

pe = positional_encoding(10, 32)
print(pe.shape)

(10, 32)


In [ ]:
import tensorflow as tf

seq_len=5

mask=1-tf.linalg.band_part(tf.ones((seq_len,seq_len)),-1,0)
print("Attention Mask",mask.numpy())

Attention Mask [[0. 1. 1. 1. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]


#Task 5: Multi Head Attention

Objective

Implement 4 attention heads.

Head 1 -> Grammar Head2 -> Context Head 3 -> Long Dependencies Head 4 -> Semantics

In [ ]:
mha=tf.keras.layers.MultiHeadAttention(
    num_heads=4,
    key_dim=32
)
x=tf.random.normal((2,5,32))

attn_output=mha(x,x,attention_mask=mask)
print("Attention Output",attn_output.shape)

Attention Output (2, 5, 32)


#Task 6: GPT Decoder Block

Objective

Build a Decoder Block from:

1.Masked Multi Head Attention

2.Add & Normalize

3.Feed forward Network

4.Add & Normalize

In [ ]:
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim)
        ])

        self.norm1 = tf.keras.layers.LayerNormalization()
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x):
        attn_output = self.att(x, x)

        x = self.norm1(attn_output + x)

        ffn_output = self.ffn(x)

        x = self.norm2(ffn_output + x)

        return x

#Task 7: GPT Model

Objective

Build GPT Architecture

Input->Embedding->Positional Encoding->Decoder Blocks->Linear->Softmax

In [ ]:
inputs = tf.keras.Input(shape=(None,))

x = tf.keras.layers.Embedding(vocab_size, 32)(inputs)

decoder = DecoderBlock(32, 4, 64)
x = decoder(x)   # second decoder block

outputs = tf.keras.layers.Dense(vocab_size, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 32)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_block (DecoderBlock)    │ (None, None, 32)       │        21,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, None, 16)       │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,160 (86.56 KB)

 Trainable params: 22,160 (86.56 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

X = []
y = []

for inp, target in sequences:
    X.append(inp)
    y.append(target)

X = pad_sequences(X, padding='pre')
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (14, 4)
y shape: (14,)


In [ ]:
print(model.output_shape)

(None, None, 16)


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
inputs = tf.keras.Input(shape=(None,))

x = tf.keras.layers.Embedding(vocab_size, 32)(inputs)

x = DecoderBlock(32, 4, 64)(x)
x = DecoderBlock(32, 4, 64)(x)

# Keep only the last token representation
x = tf.keras.layers.Lambda(lambda t: t[:, -1, :])(x)

outputs = tf.keras.layers.Dense(vocab_size, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(model.output_shape)

(None, 16)


In [ ]:
history = model.fit(
    X,
    y,
    epochs=200,
    batch_size=2,
    verbose=1
)

Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.0000e+00 - loss: 3.8711
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2143 - loss: 2.2868 
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7143 - loss: 1.7562 
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7857 - loss: 1.3044 
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7857 - loss: 0.9397 
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9286 - loss: 0.7243 
Epoch 7/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9286 - loss: 0.5567 
Epoch 8/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7857 - loss: 0.4931 
Epoch 9/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8571 - loss: 0.4197 
Epoch 10/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9286 - loss: 0.3557 
Epoch 11/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9286 - loss: 0.2990 
Epoch 12/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8571 

In [ ]:
loss, acc = model.evaluate(X, y)

print("Loss:", loss)
print("Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9286 - loss: 0.1039
Loss: 0.10391117632389069
Accuracy: 0.9285714030265808


In [ ]:
sentence = "The dog barked"

tokens = tokenizer.texts_to_sequences([sentence])[0]

input_tokens = np.array([tokens])

predictions = model.predict(input_tokens, verbose=0)

predicted_token_id = np.argmax(predictions[0])

reverse_word_index = {
    v: k for k, v in tokenizer.word_index.items()
}

predicted_word = reverse_word_index.get(predicted_token_id, "<unk>")

print("Predicted Next Word:", predicted_word)

Predicted Next Word: loudly
